# Phase 1 optimizer -- backend cross-validation (SLSQP vs IPOPT vs ForwardSimulator)

This is one of 7 notebooks in `notebooks/phase1_optimization/`.
Equation/section citations here follow `docs/tt pacing optimizer.md`, the
project's canonical numbering, written as "(design-doc Eq. N)" /
"(design-doc Section N.M)"; display-unit conventions (km/min/km-h⁻¹) are
implemented in `phase_1_0_common.py`'s `as_km`/`as_min`/`as_kmh` helpers, shared
by all 7 notebooks. This one covers
**cross-backend and cross-simulator correctness checks, independent of the solver's own success flag**.

It is fully self-contained: every rider/course/baseline it needs is
rebuilt here via `phase_1_0_common.py` (shared by all 7 split notebooks),
so it runs standalone from a fresh kernel without any other notebook
having run first. Courses used: flat, rolling (synthetic) + Giro10, TdF16, TARA (real). Approx. runtime: ~7.2 min (measured).

In [1]:
import sys
sys.path.insert(0, ".")
import phase_1_0_common as pc

pc.print_rider_summary()

reference_rider        mass= 72.0 kg  CP= 280.0 W  W'= 20.0 kJ  CdA=0.25 m^2  P_max=   900 W
evenepoel_like_rider   mass= 63.5 kg  CP= 425.0 W  W'= 20.0 kJ  CdA=0.21 m^2  P_max=  1400 W
ganna_like_rider       mass= 82.0 kg  CP= 480.0 W  W'= 20.0 kJ  CdA=0.19 m^2  P_max=  1600 W


In [2]:
courses = pc.load_real_courses()
giro10_course, tdf16_course, tara_course = courses["giro10"], courses["tdf16"], courses["tara"]
pc.print_real_course_summary(courses)

Giro 2026 Stage 10   n_nodes= 800  smoothing_length_m=  150 m  mean|grade|= 0.34%  max|grade|= 2.05%
TdF 2026 Stage 16    n_nodes= 800  smoothing_length_m=  400 m  mean|grade|= 3.37%  max|grade|= 7.59%
TARA 2026 Stage 3    n_nodes= 800  smoothing_length_m=  250 m  mean|grade|= 2.42%  max|grade|= 6.22%


## Backend cross-validation

The commit's actual correctness argument is not "the solver reported
success". That flag was uninformative for two separate reasons, one now
fixed and one not: SLSQP's `ftol` was unreachable, so it always exited on
the iteration limit (issue #6 item 7.6, fixed), and IPOPT never converges
on this problem at all (open). Instead, two independent checks are used:

1. **Cross-backend agreement**: SLSQP and IPOPT are different NLP
   algorithms (SLSQP: sequential quadratic programming; IPOPT: interior
   point) consuming the *same* analytic objective/constraint/Jacobian
   from `CollocationProblem`. If they agree closely on the optimal time,
   that is strong evidence the transcription and gradients are correct,
   independent of either solver's internal convergence bookkeeping.
2. **Independent re-simulation**: the NLP's own power plan, replayed
   through `ForwardSimulator` (a completely separate integrator, RK4 in
   the distance domain), should reproduce essentially the same total
   time.

Both are demonstrated below on two synthetic courses (a near-flat one and
a gently rolling one) and three real GPX courses (2.3-2.5), each course's
own Hermite-Simpson/SLSQP baseline rebuilt here as the SLSQP side of the
comparison.

### 2.1 Flat course

In [3]:
flat_course = pc.build_flat_course()
opt_hs_flat, res_hs_flat = pc.build_hs_baseline(pc.reference_rider, flat_course, pc.calm_wind, pc.N_INTERVALS_BASIC)

rel_diff_backend_flat, rel_diff_sim_flat = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, flat_course, pc.calm_wind, res_hs_flat, pc.N_INTERVALS_BASIC, "flat"
)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************



[flat] SLSQP: T = 57.1178 min  (success=True)
[flat] IPOPT: T = 57.1192 min  (success=False)
[flat] Relative difference: 0.000024  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[flat] NLP (SLSQP) time:      57.1178 min
[flat] ForwardSimulator time: 57.1883 min
[flat] Relative difference: 0.001233  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


### 2.2 Rolling course

In [4]:
rolling_course = pc.build_rolling_course()
opt_hs_rolling, res_hs_rolling = pc.build_hs_baseline(pc.reference_rider, rolling_course, pc.calm_wind, pc.N_INTERVALS_BASIC)

rel_diff_backend_rolling, rel_diff_sim_rolling = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, rolling_course, pc.calm_wind, res_hs_rolling, pc.N_INTERVALS_BASIC, "rolling"
)

[rolling] SLSQP: T = 34.0745 min  (success=True)
[rolling] IPOPT: T = 34.0779 min  (success=False)
[rolling] Relative difference: 0.000099  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[rolling] NLP (SLSQP) time:      34.0745 min
[rolling] ForwardSimulator time: 34.1752 min
[rolling] Relative difference: 0.002954  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


Both checks pass comfortably inside their test gates on both courses.
Note the asymmetry in the flags above: SLSQP reports `success=True`, IPOPT
`success=False`, on every course in this notebook. That is not a quality
difference between the two answers -- they agree to 0.002% on the flat
course. It is that IPOPT genuinely never converges here: it exits
`Maximum_Iterations_Exceeded` on all five courses with an overall NLP
error around 1.2e+01, essentially all of it dual infeasibility, and no
tolerance setting changes the returned iterate (tracked as its own issue,
suspected to be the non-smooth `P = CP` kink in `DifferentialModel`).
SLSQP's flag became meaningful only with issue #6 item 7.6.

So the design decision stands, for a sharper reason than before: gate
correctness on cross-backend agreement and independent re-simulation, not
on the solver's own flag -- here one backend's flag is uninformative by
construction.

### 2.3 Giro 2026 Stage 10 (real, flat)

In [5]:
opt_hs_giro10, res_hs_giro10 = pc.build_hs_baseline(pc.ganna_like_rider, giro10_course, pc.calm_wind, pc.N_INTERVALS_GIRO10)

rel_diff_backend_giro10, rel_diff_sim_giro10 = pc.run_backend_and_sim_crosscheck(
    pc.ganna_like_rider, giro10_course, pc.calm_wind, res_hs_giro10, pc.N_INTERVALS_GIRO10, "giro10"
)

[giro10] SLSQP: T = 43.2630 min  (success=True)
[giro10] IPOPT: T = 43.2742 min  (success=False)
[giro10] Relative difference: 0.000260  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[giro10] NLP (SLSQP) time:      43.2630 min
[giro10] ForwardSimulator time: 43.2549 min
[giro10] Relative difference: 0.000186  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


### 2.4 TdF 2026 Stage 16 (real, mountainous)

The steep-grade, hairpin-heavy course is the sternest test in this notebook for whether SLSQP and IPOPT still agree away from gentle terrain.

In [6]:
opt_hs_tdf16, res_hs_tdf16 = pc.build_hs_baseline(pc.evenepoel_like_rider, tdf16_course, pc.calm_wind, pc.N_INTERVALS_TDF16)

rel_diff_backend_tdf16, rel_diff_sim_tdf16 = pc.run_backend_and_sim_crosscheck(
    pc.evenepoel_like_rider, tdf16_course, pc.calm_wind, res_hs_tdf16, pc.N_INTERVALS_TDF16, "tdf16"
)

[tdf16] SLSQP: T = 32.2492 min  (success=True)
[tdf16] IPOPT: T = 32.1668 min  (success=False)
[tdf16] Relative difference: 0.002556  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[tdf16] NLP (SLSQP) time:      32.2492 min
[tdf16] ForwardSimulator time: 32.2544 min
[tdf16] Relative difference: 0.000161  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


### 2.5 TARA 2026 Stage 3 (real, rolling/TTT)

In [7]:
opt_hs_tara, res_hs_tara = pc.build_hs_baseline(pc.reference_rider, tara_course, pc.calm_wind, pc.N_INTERVALS_TARA)

rel_diff_backend_tara, rel_diff_sim_tara = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, tara_course, pc.calm_wind, res_hs_tara, pc.N_INTERVALS_TARA, "tara"
)

[tara] SLSQP: T = 42.8427 min  (success=True)
[tara] IPOPT: T = 42.9193 min  (success=False)
[tara] Relative difference: 0.001787  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[tara] NLP (SLSQP) time:      42.8427 min
[tara] ForwardSimulator time: 42.7793 min
[tara] Relative difference: 0.001480  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


Real terrain no longer breaks either gate the way it did. **The
independent-re-simulation check now passes on all three real courses**:
Giro10 0.019%, TdF16 0.016%, TARA 0.148%, against the 0.5% test gate.
TARA used to miss that gate by more than 5x (2.68%).

The cross-backend check is the one that still does not fully clear its
much tighter <0.1% bound (which, per `tests/test_phase_1.py`, is only ever
asserted against the synthetic flat-course fixture). **Giro10 now passes
it outright at 0.026%** -- the first real course ever to do so -- while
TdF16 sits at 0.256% and TARA at 0.179%.

Both movements come from issue #6, and the TARA case is worth stating
plainly because this section previously got it wrong. It read:

> TARA's disagreement is large enough ... that SLSQP is plausibly landing
> on a materially different local solution on this course than IPOPT does.

That hypothesis was never tested here, and it was wrong. Both solutions
were defect-feasible; what differed was the *mesh*. `_graded_mesh` let its
geometric width ramp overshoot the uniform interval width, putting a
single ~1.5 km Hermite-Simpson interval in the middle of real terrain a
few hundred metres into the course, and because the ramp was also capped
at 9 intervals regardless of `n_intervals`, mesh refinement never repaired
it. The two backends then refined differently and optimized two
differently-wrong versions of the course. Capping the ramp (item 7.1)
takes TARA's cross-backend disagreement from 5.96% to 0.179%, a factor of
33, and its simulator disagreement from 2.68% to 0.148%.

One caveat this section should have carried from the start, and now does:
`run_backend_and_sim_crosscheck` lets each backend run its own
defect-driven refinement loop, so the two can finish on different meshes
and the comparison is then not like with like. `OptimizationResult.s_m` is
authoritative for the achieved mesh (`n_intervals` reports it too, since
#6 item 7.3). Holding the mesh fixed is issue #6 item 7.2 and is not done
here yet.

(The parenthetical smoothing-length sensitivity sweep this cell used to
carry -- 3.58% at 150 m, 2.24% at 300 m -- was measured off-notebook on
pre-#5 courses and is not reproduced by any cell here, so it has been
dropped rather than restated from memory.)